In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_1.py ──
"""
Shared infrastructure for MLFP02 Exercise 1 — Probability and Bayesian
Fundamentals.

Contains: HDB data loading (4-ROOM 2020+ slice), prior/posterior math for
the Normal-Normal and Beta-Binomial conjugate families, bootstrap helpers,
and output directory wiring.

Technique-specific narrative (MLE derivation, Bayes scenarios, credible vs
confidence simulation, bootstrap comparison) does NOT belong here — it
lives in the per-technique files.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

# Output directory for plots (HTML) — every technique writes here.
OUTPUT_DIR = Path("outputs") / "mlfp02_ex1_bayes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical prior used across all four technique files. A single source
# of truth means Task 5's prior sweep and Task 4's posterior use the same
# starting point.
PRIOR_MU_0: float = 500_000.0  # SGD — Singapore 4-room HDB market anchor
PRIOR_SIGMA_0: float = 100_000.0  # SGD — moderate uncertainty

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flats (Singapore, data.gov.sg)
# ════════════════════════════════════════════════════════════════════════

_loader = MLFPDataLoader()


def load_hdb_all() -> pl.DataFrame:
    """Full HDB resale dataset filtered to 2020+ transactions.

    Returns a polars DataFrame with columns: month, town, flat_type,
    resale_price, plus any others present in the source parquet. Used
    by Task 1 (truth tables) and Task 8 (expected value by flat type).
    """
    hdb = _loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def load_hdb_4room() -> pl.DataFrame:
    """4-ROOM slice of HDB resale (2020+) — the primary estimation target."""
    return load_hdb_all().filter(pl.col("flat_type") == "4 ROOM")


def load_hdb_prices_4room() -> np.ndarray:
    """Return the 4-room resale_price column as a float64 numpy array.

    This is the primary observation vector for MLE, Normal-Normal, and
    bootstrap tasks.
    """
    return load_hdb_4room()["resale_price"].to_numpy().astype(np.float64)


# ════════════════════════════════════════════════════════════════════════
# NORMAL-NORMAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalPosterior:
    """Posterior for μ under a Normal-Normal conjugate with known σ."""

    mean: float
    std: float
    prior_mean: float
    prior_std: float
    n: int
    sigma_known: float

    @property
    def precision_prior(self) -> float:
        return 1.0 / self.prior_std**2

    @property
    def precision_data(self) -> float:
        return self.n / self.sigma_known**2

    @property
    def prior_weight(self) -> float:
        """Fraction of posterior precision contributed by the prior (0..1)."""
        return self.precision_prior / (self.precision_prior + self.precision_data)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        z = stats.norm.ppf(0.5 + level / 2)
        return (self.mean - z * self.std, self.mean + z * self.std)


def normal_normal_posterior(
    data: np.ndarray,
    prior_mean: float,
    prior_std: float,
    sigma_known: float,
) -> NormalPosterior:
    """Closed-form posterior for μ under N(μ₀, σ₀²) prior and known σ.

    Posterior precision = prior precision + n / σ². Posterior mean is the
    precision-weighted average of the prior mean and the sample mean.
    """
    n = len(data)
    xbar = float(np.mean(data))
    prec_prior = 1.0 / prior_std**2
    prec_data = n / sigma_known**2
    prec_post = prec_prior + prec_data
    sigma_post_sq = 1.0 / prec_post
    mu_post = sigma_post_sq * (prior_mean * prec_prior + n * xbar / sigma_known**2)
    return NormalPosterior(
        mean=mu_post,
        std=float(np.sqrt(sigma_post_sq)),
        prior_mean=prior_mean,
        prior_std=prior_std,
        n=n,
        sigma_known=sigma_known,
    )


# ════════════════════════════════════════════════════════════════════════
# BETA-BINOMIAL CONJUGATE
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class BetaPosterior:
    """Posterior for p under a Beta-Binomial conjugate."""

    alpha: float
    beta: float
    prior_alpha: float
    prior_beta: float
    k: int
    n: int

    @property
    def mean(self) -> float:
        return self.alpha / (self.alpha + self.beta)

    @property
    def prior_mean(self) -> float:
        return self.prior_alpha / (self.prior_alpha + self.prior_beta)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        lo = (1 - level) / 2
        hi = 1 - lo
        return tuple(stats.beta.ppf([lo, hi], self.alpha, self.beta).tolist())


def beta_binomial_posterior(
    k: int, n: int, prior_alpha: float, prior_beta: float
) -> BetaPosterior:
    """Closed-form posterior for p under Beta(α, β) prior and k/n Binomial."""
    return BetaPosterior(
        alpha=prior_alpha + k,
        beta=prior_beta + (n - k),
        prior_alpha=prior_alpha,
        prior_beta=prior_beta,
        k=k,
        n=n,
    )


# ════════════════════════════════════════════════════════════════════════
# MLE + CRAMER-RAO
# ════════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class NormalMLE:
    mean: float
    mle_std: float  # ddof=0, biased
    unbiased_std: float  # ddof=1, Bessel's correction
    n: int

    @property
    def fisher_information(self) -> float:
        return self.n / self.mle_std**2

    @property
    def cramer_rao_bound(self) -> float:
        return 1.0 / self.fisher_information

    @property
    def standard_error(self) -> float:
        return float(np.sqrt(self.cramer_rao_bound))


def normal_mle(data: np.ndarray) -> NormalMLE:
    """MLE for Normal (μ, σ²) with Cramér-Rao bound bookkeeping."""
    arr = np.asarray(data, dtype=np.float64)
    return NormalMLE(
        mean=float(arr.mean()),
        mle_std=float(arr.std(ddof=0)),
        unbiased_std=float(arr.std(ddof=1)),
        n=len(arr),
    )


# ════════════════════════════════════════════════════════════════════════
# BOOTSTRAP INTERVALS
# ════════════════════════════════════════════════════════════════════════


def bootstrap_mean_distribution(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42
) -> np.ndarray:
    """Non-parametric bootstrap of the sample mean."""
    rng = np.random.default_rng(seed)
    n = len(data)
    return np.array(
        [rng.choice(data, size=n, replace=True).mean() for _ in range(n_bootstrap)]
    )


def percentile_ci(draws: np.ndarray, level: float = 0.95) -> tuple[float, float]:
    lo = (1 - level) / 2 * 100
    hi = (1 + level) / 2 * 100
    return float(np.percentile(draws, lo)), float(np.percentile(draws, hi))


def bca_ci(
    data: np.ndarray, n_bootstrap: int = 10_000, seed: int = 42, level: float = 0.95
) -> tuple[float, float]:
    """Bias-corrected accelerated bootstrap CI for the mean (via scipy)."""
    result = stats.bootstrap(
        (np.asarray(data, dtype=np.float64),),
        statistic=np.mean,
        n_resamples=n_bootstrap,
        confidence_level=level,
        method="BCa",
        random_state=seed,
    )
    return float(result.confidence_interval.low), float(result.confidence_interval.high)


# ════════════════════════════════════════════════════════════════════════
# FORMATTING
# ════════════════════════════════════════════════════════════════════════


def fmt_money(x: float) -> str:
    return f"${x:,.0f}"


def print_interval(label: str, lo: float, hi: float) -> None:
    print(
        f"  {label:<28} [{fmt_money(lo)}, {fmt_money(hi)}]  width={fmt_money(hi - lo)}"
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 1.3: Conjugate Priors — Normal-Normal and
#                         Beta-Binomial
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement the Normal-Normal conjugate: prior + likelihood → posterior
#     analytically (no MCMC needed)
#   - Understand precision weighting — how data overwhelms the prior
#   - Run prior sensitivity analysis: sweep μ₀ and σ₀ to show robustness
#   - Implement the Beta-Binomial conjugate for proportions
#   - Compare weak vs strong priors and see both converge with large n
#
# PREREQUISITES: Complete 02_bayes_theorem.py (Bayes' theorem intuition)
#
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Theory — conjugate families and why they matter
#   2. Build — Normal-Normal posterior for 4-room HDB prices
#   3. Train — prior sensitivity sweep (μ₀ and σ₀ grids)
#   4. Visualise — prior vs posterior density + sensitivity heatmap
#   5. Apply — Beta-Binomial for HDB transaction success rates
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats



## THEORY — Conjugate Families and Why They Matter


In [ ]:
# A conjugate prior is a prior distribution that, when combined with a
# specific likelihood, yields a posterior in the SAME family as the prior.
# This means the posterior has a closed-form solution — no sampling needed.
#
# Normal-Normal conjugate:
#   Prior:      μ ~ N(μ₀, σ₀²)          — what we believe before data
#   Likelihood: xᵢ | μ ~ N(μ, σ²)       — how data is generated
#   Posterior:  μ | data ~ N(μₙ, σₙ²)   — updated belief
#
#   The posterior precision = prior precision + data precision:
#     1/σₙ² = 1/σ₀² + n/σ²
#   The posterior mean is a precision-weighted average:
#     μₙ = σₙ² × (μ₀/σ₀² + n×x̄/σ²)
#
# Beta-Binomial conjugate:
#   Prior:      p ~ Beta(α, β)           — belief about a proportion
#   Likelihood: k | p ~ Binomial(n, p)   — count of successes
#   Posterior:  p | data ~ Beta(α+k, β+n-k)
#
# Why this matters: in practice, you rarely have infinite data. Conjugate
# priors let you combine domain knowledge with limited data in a
# principled, computationally free way. No MCMC, no GPU, no tuning.



## TASK 2 — BUILD: Normal-Normal Posterior for 4-Room HDB Prices


In [ ]:
print("\n" + "=" * 70)
print("  MLFP02 Exercise 1.3: Conjugate Priors")
print("=" * 70)

prices = load_hdb_prices_4room()
mle = normal_mle(prices)

# Use the canonical prior from shared module
mu_0 = PRIOR_MU_0
sigma_0 = PRIOR_SIGMA_0

# Compute posterior using the shared helper
posterior = normal_normal_posterior(
    data=prices,
    prior_mean=mu_0,
    prior_std=sigma_0,
    sigma_known=mle.mle_std,
)

print(f"\n=== Normal-Normal Conjugate Posterior ===")
print(f"Prior: μ ~ N(μ₀={fmt_money(mu_0)}, σ₀={fmt_money(sigma_0)})")
print(f"Likelihood: X|μ ~ N(μ, σ²) with σ={fmt_money(mle.mle_std)} (plug-in)")
print(
    f"\nPosterior: μ|data ~ N(μₙ={fmt_money(posterior.mean)}, σₙ={posterior.std:,.2f})"
)
print(f"Prior precision:     {posterior.precision_prior:.2e}")
print(f"Data precision:      {posterior.precision_data:.2e}")
print(
    f"Posterior precision:  {posterior.precision_prior + posterior.precision_data:.2e}"
)
print(
    f"Data-to-prior precision ratio: "
    f"{posterior.precision_data / posterior.precision_prior:.0f}x"
)
print(
    f"  → Posterior is dominated by "
    f"{'data' if posterior.precision_data > posterior.precision_prior else 'prior'}"
)

# 95% credible interval
ci_lo, ci_hi = posterior.credible_interval(0.95)
print(f"\n95% Bayesian credible interval: [{fmt_money(ci_lo)}, {fmt_money(ci_hi)}]")
# INTERPRETATION: A credible interval has a direct probability statement:
# "Given the data, there is a 95% probability that the true mean price
# lies in this range." This is STRONGER than the frequentist CI, which
# says "95% of intervals constructed this way would contain the true mean."



### Checkpoint 1


In [ ]:
assert posterior.precision_data > 0, "Data precision must be positive"
assert posterior.std < sigma_0, "Posterior std should be narrower than prior"
assert ci_lo < posterior.mean < ci_hi, "Posterior mean must be within its own CI"
assert (
    abs(posterior.mean - mle.mean) < sigma_0
), "Posterior should be near MLE with large n"
print("\n✓ Checkpoint 1 passed — Normal-Normal posterior computed\n")



## TASK 3 — TRAIN: Prior Sensitivity Sweep


In [ ]:
# How does the posterior change as we vary the prior?
# Sweep prior mean from $300K to $700K and prior std from $20K to $200K.
# This shows when the prior matters and when data overwhelms it.

print("=== Prior Sensitivity Analysis ===")

# Sweep prior mean
print(f"\n--- Varying prior mean (σ₀ = {fmt_money(sigma_0)} fixed) ---")
print(
    f"{'Prior μ₀':>12} {'Posterior μₙ':>15} {'Shift from MLE':>16} {'Prior Weight':>13}"
)
print("─" * 60)
mu_sweep_values = [300_000, 400_000, 500_000, 600_000, 700_000]
for mu_sweep in mu_sweep_values:
    post_sweep = normal_normal_posterior(prices, mu_sweep, sigma_0, mle.mle_std)
    print(
        f"${mu_sweep:>10,.0f}  ${post_sweep.mean:>13,.0f}  "
        f"{post_sweep.mean - mle.mean:>+14,.0f}  "
        f"{post_sweep.prior_weight * 100:>11.4f}%"
    )

# Sweep prior std
print(f"\n--- Varying prior std (μ₀ = {fmt_money(mu_0)} fixed) ---")
print(f"{'Prior σ₀':>12} {'Posterior μₙ':>15} {'Prior Weight':>13}")
print("─" * 45)
sigma_sweep_values = [20_000, 50_000, 100_000, 200_000, 500_000]
for sigma_sweep in sigma_sweep_values:
    post_sweep = normal_normal_posterior(prices, mu_0, sigma_sweep, mle.mle_std)
    print(
        f"${sigma_sweep:>10,.0f}  ${post_sweep.mean:>13,.0f}  "
        f"{post_sweep.prior_weight * 100:>11.4f}%"
    )
# INTERPRETATION: Even a very opinionated prior (σ₀=$20K) gets overwhelmed
# by the data when n is large. But with small n, the prior choice matters.
# A sensitivity analysis like this should accompany any Bayesian report
# to show stakeholders that conclusions are robust to prior assumptions.



### Checkpoint 2


In [ ]:
# Verify that varying the prior mean within ±$200K barely moves posterior
for mu_test in [300_000, 700_000]:
    post_test = normal_normal_posterior(prices, mu_test, sigma_0, mle.mle_std)
    assert (
        abs(post_test.mean - mle.mean) < 5000
    ), "With large n, posterior should be near MLE regardless of prior mean"
print("\n✓ Checkpoint 2 passed — prior sensitivity demonstrates data dominance\n")



## TASK 4 — VISUALISE: Prior vs Posterior + Sensitivity Heatmap


In [ ]:
# -- Plot 1: Prior vs Posterior distributions --
x_prior = np.linspace(mu_0 - 3 * sigma_0, mu_0 + 3 * sigma_0, 500)
prior_pdf = stats.norm.pdf(x_prior, mu_0, sigma_0)

x_post = np.linspace(
    posterior.mean - 5 * posterior.std, posterior.mean + 5 * posterior.std, 500
)
posterior_pdf = stats.norm.pdf(x_post, posterior.mean, posterior.std)

fig1 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Prior Distribution", "Posterior Distribution"],
)
fig1.add_trace(
    go.Scatter(
        x=x_prior,
        y=prior_pdf,
        name=f"Prior N({fmt_money(mu_0)}, {fmt_money(sigma_0)}²)",
        line={"color": "blue"},
    ),
    row=1,
    col=1,
)
fig1.add_trace(
    go.Scatter(
        x=x_post,
        y=posterior_pdf,
        name=f"Posterior N({fmt_money(posterior.mean)}, {posterior.std:,.0f}²)",
        line={"color": "red"},
    ),
    row=1,
    col=2,
)
fig1.update_layout(
    title="Normal-Normal Conjugate: Prior vs Posterior for 4-Room HDB Mean",
    height=400,
)
fig1.write_html(str(OUTPUT_DIR / "prior_vs_posterior.html"))
print("Saved: prior_vs_posterior.html")

# -- Plot 1b: prior × likelihood = posterior on ONE axis --
# The prior-vs-posterior pair hides the third actor: the LIKELIHOOD
# L(μ) = Π N(xᵢ | μ, σ̂), the data's own voice. On the full sample the
# likelihood is a needle (SE = σ/√n is tiny), so for this panel only we
# use a seeded n=50 subsample — its likelihood is wide enough to see
# alongside the prior. All three curves are peak-normalised so SHAPE and
# POSITION (not area) carry the comparison.
rng_lik = np.random.default_rng(42)
small = rng_lik.choice(prices, size=50, replace=False)
sigma_hat = mle.mle_std
mu_grid_lik = np.linspace(300_000, 700_000, 600)
loglik_grid = np.array(
    [np.sum(stats.norm.logpdf(small, loc=mu, scale=sigma_hat)) for mu in mu_grid_lik]
)
lik_curve = np.exp(loglik_grid - loglik_grid.max())

prior_curve = stats.norm.pdf(mu_grid_lik, mu_0, sigma_0)
prior_curve = prior_curve / prior_curve.max()

post_small = normal_normal_posterior(small, mu_0, sigma_0, sigma_hat)
post_curve = stats.norm.pdf(mu_grid_lik, post_small.mean, post_small.std)
post_curve = post_curve / post_curve.max()

fig1b = go.Figure()
fig1b.add_trace(
    go.Scatter(
        x=mu_grid_lik,
        y=prior_curve,
        name=f"Prior N({fmt_money(mu_0)}, {fmt_money(sigma_0)}²)",
        line={"color": "blue", "dash": "dash"},
    )
)
fig1b.add_trace(
    go.Scatter(
        x=mu_grid_lik,
        y=lik_curve,
        name=f"Likelihood (n=50, x̄={fmt_money(float(small.mean()))})",
        line={"color": "green"},
    )
)
fig1b.add_trace(
    go.Scatter(
        x=mu_grid_lik,
        y=post_curve,
        name=f"Posterior N({fmt_money(post_small.mean)}, {post_small.std:,.0f}²)",
        line={"color": "red"},
    )
)
fig1b.update_layout(
    title=(
        "Prior × Likelihood = Posterior (n=50 subsample, "
        "peak-normalised — shapes comparable, areas not)"
    ),
    xaxis_title="μ ($)",
    yaxis_title="Peak-normalised density / likelihood",
    height=420,
)
fig1b.write_html(str(OUTPUT_DIR / "prior_likelihood_posterior.html"))
print("Saved: prior_likelihood_posterior.html")
# INTERPRETATION: the posterior sits BETWEEN the prior and the
# likelihood, pulled toward whichever has more precision (1/variance).
# With n=50 the likelihood dominates but the prior still tugs the
# posterior mean a little toward $500K; with the full sample (fig1) the
# tug is invisible. The likelihood's width is σ/√n — that shrinkage is
# the whole story of "data overwhelming the prior".

# -- Plot 2: Sensitivity heatmap (μ₀ × σ₀ → posterior mean) --
mu_grid = np.linspace(300_000, 700_000, 20)
sigma_grid = np.linspace(20_000, 300_000, 20)
z_sensitivity = np.zeros((len(sigma_grid), len(mu_grid)))

for i, sg in enumerate(sigma_grid):
    for j, mg in enumerate(mu_grid):
        post_g = normal_normal_posterior(prices, mg, sg, mle.mle_std)
        z_sensitivity[i, j] = post_g.mean

fig2 = go.Figure(
    data=go.Heatmap(
        z=z_sensitivity,
        x=mu_grid,
        y=sigma_grid,
        colorscale="RdBu_r",
        colorbar={"title": "Posterior μₙ ($)"},
    )
)
fig2.update_layout(
    title="Prior Sensitivity: Posterior Mean as f(μ₀, σ₀)",
    xaxis_title="Prior Mean μ₀ ($)",
    yaxis_title="Prior Std σ₀ ($)",
    height=500,
)
fig2.write_html(str(OUTPUT_DIR / "sensitivity_heatmap.html"))
print("Saved: sensitivity_heatmap.html")



### Checkpoint 3


In [ ]:
assert z_sensitivity.shape == (20, 20), "Heatmap should be 20×20"
_lo, _hi = min(mu_0, float(small.mean())), max(mu_0, float(small.mean()))
assert _lo <= post_small.mean <= _hi, (
    "Posterior mean must lie between the prior mean and the subsample mean "
    "(precision-weighted average)"
)
assert abs(lik_curve.max() - 1.0) < 1e-12, "Likelihood curve is peak-normalised"
print("\n✓ Checkpoint 3 passed — visualisations saved\n")



## TASK 5 — APPLY: Beta-Binomial for HDB Transaction Success Rates


In [ ]:
# Different conjugate family: Beta prior for a proportion.
# Use case: what fraction of 4-room HDB transactions close above $500K?
#
# Prior: p ~ Beta(α, β) where E[p] = α/(α+β)
# Likelihood: k successes in n trials ~ Binomial(n, p)
# Posterior: p | data ~ Beta(α + k, β + n - k)
#
# Business context: a property fund needs to know the "success rate"
# of 4-room flats closing above $500K to set portfolio thresholds.
# If the rate drops below 40%, the fund reallocates to 3-room inventory.

print("=== Beta-Binomial Conjugate: Transaction Success Rates ===")

threshold = 500_000
k_success = int((prices > threshold).sum())
n_trials = len(prices)
empirical_rate = k_success / n_trials

print(f"Threshold: {fmt_money(threshold)}")
print(
    f"Successes (price > threshold): {k_success:,} / {n_trials:,} = {empirical_rate:.2%}"
)

# Weak prior: Beta(2, 2) — slight preference for 50%
post_weak = beta_binomial_posterior(
    k_success, n_trials, prior_alpha=2.0, prior_beta=2.0
)

# Strong prior: Beta(20, 80) — 20% success rate expected
post_strong = beta_binomial_posterior(
    k_success, n_trials, prior_alpha=20.0, prior_beta=80.0
)

ci_weak = post_weak.credible_interval(0.95)
ci_strong = post_strong.credible_interval(0.95)

print(f"\n--- Weak Prior: Beta(2, 2), E[p]={post_weak.prior_mean:.2f} ---")
print(f"Posterior: Beta({post_weak.alpha:.0f}, {post_weak.beta:.0f})")
print(f"Posterior mean: {post_weak.mean:.4f} ({post_weak.mean:.2%})")
print(f"95% CI: [{ci_weak[0]:.4f}, {ci_weak[1]:.4f}]")

print(f"\n--- Strong Prior: Beta(20, 80), E[p]={post_strong.prior_mean:.2f} ---")
print(f"Posterior: Beta({post_strong.alpha:.0f}, {post_strong.beta:.0f})")
print(f"Posterior mean: {post_strong.mean:.4f} ({post_strong.mean:.2%})")
print(f"95% CI: [{ci_strong[0]:.4f}, {ci_strong[1]:.4f}]")

print(f"\nEmpirical rate: {empirical_rate:.4f}")
print(
    f"Both posteriors converge toward {empirical_rate:.2%} because n={n_trials:,} is large."
)
# INTERPRETATION: The Beta-Binomial is the natural Bayesian model for
# proportions. Even a strong prior (Beta(20,80) suggesting 20%) gets
# overwhelmed by thousands of observations.

# Beta-Binomial visualisation
x_beta = np.linspace(0, 1, 500)
beta_prior_pdf = stats.beta.pdf(x_beta, 2.0, 2.0)
beta_post_pdf = stats.beta.pdf(x_beta, post_weak.alpha, post_weak.beta)
beta_strong_pdf = stats.beta.pdf(x_beta, post_strong.alpha, post_strong.beta)

fig3 = go.Figure()
fig3.add_trace(
    go.Scatter(
        x=x_beta,
        y=beta_prior_pdf,
        name="Prior Beta(2,2)",
        line={"color": "blue", "dash": "dash"},
    )
)
fig3.add_trace(
    go.Scatter(
        x=x_beta,
        y=beta_post_pdf,
        name="Posterior (weak prior)",
        line={"color": "red"},
    )
)
fig3.add_trace(
    go.Scatter(
        x=x_beta,
        y=beta_strong_pdf,
        name="Posterior (strong prior)",
        line={"color": "orange", "dash": "dot"},
    )
)
fig3.add_vline(
    x=empirical_rate,
    line_dash="dot",
    annotation_text=f"Empirical: {empirical_rate:.2%}",
)
fig3.update_layout(
    title="Beta-Binomial: P(price > $500K) — Prior vs Posterior",
    xaxis_title="Proportion",
    yaxis_title="Density",
    height=450,
)
fig3.write_html(str(OUTPUT_DIR / "beta_binomial.html"))
print("Saved: beta_binomial.html")

# Dollar impact for property fund
fund_threshold = 0.40
print(f"\n--- Fund Decision: rebalance if rate < {fund_threshold:.0%} ---")
p_below_threshold = stats.beta.cdf(fund_threshold, post_weak.alpha, post_weak.beta)
print(f"P(rate < {fund_threshold:.0%}) = {p_below_threshold:.6f}")
if p_below_threshold < 0.05:
    print("→ Very unlikely the rate is below 40%. Keep 4-room allocation.")
else:
    print("→ Non-trivial probability of rate < 40%. Consider rebalancing.")



### Checkpoint 4


In [ ]:
assert post_weak.alpha == 2.0 + k_success, "Posterior alpha update incorrect"
assert post_weak.beta == 2.0 + (n_trials - k_success), "Posterior beta update incorrect"
assert ci_weak[0] < post_weak.mean < ci_weak[1], "Posterior mean must be within CI"
assert (
    abs(post_weak.mean - empirical_rate) < 0.01
), "With weak prior and large n, posterior mean should be near empirical rate"
print("\n✓ Checkpoint 4 passed — Beta-Binomial conjugate and fund decision complete\n")



## REFLECTION


✓ Normal-Normal conjugate: prior + likelihood → closed-form posterior
  ✓ The likelihood curve IS the data's voice: L(μ) = Π N(xᵢ | μ, σ̂),
    width σ/√n — and the posterior is the precision-weighted compromise
    between it and the prior
  ✓ Precision weighting: posterior precision = prior + data precision
  ✓ Prior sensitivity: sweeping μ₀ and σ₀ proves conclusions are
    robust — with large n, prior barely moves the posterior
  ✓ Beta-Binomial conjugate: natural model for proportions, same
    "prior + data → posterior" logic for success rates
  ✓ Weak vs strong priors: both converge to the empirical rate when
    n is large — data overwhelms belief
  ✓ Business framing: property fund threshold decision using the
    posterior CDF — "what is P(rate < 40%)?"

  NEXT: In 04_intervals.py, you'll compare Bayesian credible intervals
  with frequentist confidence intervals, run bootstrap CIs, and apply
  Bayesian estimation across all flat types to see how prior influence
  varies with sample size.


In [ ]:
print("═" * 70)
print("  WHAT YOU'VE MASTERED (1.3 — Conjugate Priors)")
print("═" * 70)
print(
)

print("\n✓ Exercise 1.3 complete — Conjugate Priors")

